# Halftone Simulator

Simulate the full halftone printing pipeline: separate an image into CMYK, screen each channel at its classic angle, apply dot gain, and recombine subtractively on paper. Compare AM vs FM screening and watch moiré appear and disappear as you change angles.

Uses `tools/halftone_renderer.py` and `tools/halftone_common.py`.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Make the repo's tools importable from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import halftone_common as hc

def demo_image(size=256):
    """A synthetic test image: smooth ramp + gradient disk, for when you
    don't have a photo handy. Replace with hc.load_image('path.jpg')."""
    y, x = np.mgrid[0:size, 0:size] / size
    ramp = x
    disk = np.clip(1.2 - 2.0 * np.hypot(x - 0.5, y - 0.5), 0, 1)
    g = np.clip(0.5 * ramp + 0.5 * disk, 0, 1)
    return np.stack([g, np.roll(g, 20, 0), 1 - g], axis=-1)

img = demo_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title('source'); plt.axis('off'); plt.show()


## CMYK separation
Every printed color is built from cyan, magenta, yellow and black ink. Here is the separation of our source image.

In [ ]:
cmyk = hc.rgb_to_cmyk(img)
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for a, name, i in zip(ax, 'CMYK', range(4)):
    a.imshow(cmyk[..., i], cmap='gray_r'); a.set_title(name); a.axis('off')
plt.show()

## AM screening with classic angles
C:15° M:75° Y:0° K:45° — the 30° spacing that minimises moiré.

In [ ]:
import halftone_renderer as hr

class Args: pass
args = Args()
args.mode='am'; args.lpi=40; args.shape='round'; args.dot_gain=0.0
args.dot_size=1.5; args.seed=0
out = hr.render(img, args)
plt.figure(figsize=(6,6)); plt.imshow(out); plt.title('AM halftone'); plt.axis('off'); plt.show()

## Provoking moiré
Set every channel to the *same* angle and the screens beat against each other. (We monkey-patch the angle table to demonstrate.)

In [ ]:
good = dict(hr.ANGLES)
hr.ANGLES.update(C=20, M=22, Y=24, K=26)  # nearly parallel -> moire
bad = hr.render(img, args)
hr.ANGLES.update(good)                    # restore
fig, ax = plt.subplots(1, 2, figsize=(12,6))
ax[0].imshow(out); ax[0].set_title('classic angles'); ax[0].axis('off')
ax[1].imshow(bad); ax[1].set_title('near-parallel -> moire'); ax[1].axis('off')
plt.show()

## Dot gain
Ink spreads on paper, darkening midtones. Compare 0% vs 25% gain.

In [ ]:
args.dot_gain = 0.25
gain = hr.render(img, args)
args.dot_gain = 0.0
fig, ax = plt.subplots(1, 2, figsize=(12,6))
ax[0].imshow(out);  ax[0].set_title('no dot gain'); ax[0].axis('off')
ax[1].imshow(gain); ax[1].set_title('25% dot gain'); ax[1].axis('off')
plt.show()

## AM vs FM
FM (stochastic) screening has no angles and no moiré.

In [ ]:
args.mode='fm'; args.lpi=90; args.dot_size=1.5
fm = hr.render(img, args)
plt.figure(figsize=(6,6)); plt.imshow(fm); plt.title('FM / stochastic'); plt.axis('off'); plt.show()

Try next: load a real portrait with `hc.load_image(...)`, sweep `lpi` from 20 to 120, and find the threshold where the dots vanish at your screen's viewing distance.